# video-app lab 07 - FIRST REAL CLIP (Lightning weights loaded correctly)
Verified cause of the last error: the file `animatediff_lightning_4step_diffusers.safetensors` does NOT
contain LoRA pairs (`lora_A/lora_B`). It contains full motion-module weights (`down_blocks...motion_modules...`).
So `load_lora_weights` must fail - we load it as a state dict into the UNet instead.
Same free stack: SD1.5 base + AnimateDiff v1-5-2 motion adapter + Lightning 4-step weights.


In [ ]:
import os, shutil, psutil, torch
vm = psutil.virtual_memory()
print(f'RAM {vm.used/1024**3:.1f}/{vm.total/1024**3:.1f} GB | disk {shutil.disk_usage("/").free/1024**3:.0f} GB free | cuda {torch.cuda.is_available()}')
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU. Runtime > Change runtime type > T4 GPU + High-RAM > Save.')
p = torch.cuda.get_device_properties(0); print('GPU', p.name, round(p.total_memory/1024**3,1), 'GB -> OK')


In [ ]:
import gc, time, torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
DTYPE = torch.float16
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-2', torch_dtype=DTYPE, low_cpu_mem_usage=True)
pipe = AnimateDiffPipeline.from_pretrained('emilianJR/epiCRealism', motion_adapter=adapter,
        torch_dtype=DTYPE, low_cpu_mem_usage=True, use_safetensors=True)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.enable_vae_slicing(); pipe.enable_attention_slicing()
pipe.to('cuda'); gc.collect()
print('base + adapter on GPU | VRAM', round(torch.cuda.memory_allocated()/1024**3,2), 'GB')


In [ ]:
# Lightning distilled motion-module weights (not LoRA) - load into the UNet
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
STEPS = 25
try:
    w = hf_hub_download('ByteDance/AnimateDiff-Lightning', 'animatediff_lightning_4step_diffusers.safetensors')
    sd = load_file(w)
    sd = {('unet.' + k): v.to(DTYPE) for k, v in sd.items()}
    res = pipe.unet.load_state_dict(sd, strict=False)
    print('missing:', len(res.missing_keys), '| unexpected:', len(res.unexpected_keys))
    if len(res.unexpected_keys) == 0:
        STEPS = 4
        print('LIGHTNING LOADED -> 4 steps')
    else:
        print('fallback: standard 25 steps')
except Exception as e:
    print('lightning load failed ->', type(e).__name__, str(e)[:200], '| fallback 25 steps')
pipe.to('cuda')


In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
PROMPT = ('cinematic photo of a red fox trotting through fresh snow in a pine forest, '
          'morning light rays, shallow depth of field, steady camera, 35mm film')
t0 = time.time()
frames = pipe(prompt=PROMPT, negative_prompt='bad quality, blurry, distorted, watermark, text',
              num_frames=16, height=512, width=512,
              guidance_scale=1.0 if STEPS == 4 else 7.5, num_inference_steps=STEPS,
              generator=torch.Generator('cpu').manual_seed(42)).frames[0]
print('GENERATED', STEPS, 'steps in', round(time.time()-t0), 's')
print('peak VRAM GB', round(torch.cuda.max_memory_allocated()/1024**3,2))
export_to_video(frames, '/content/clip_07_fox.mp4', fps=8)
print('file MB', round(os.path.getsize('/content/clip_07_fox.mp4')/1024**2,1))
display(Video('/content/clip_07_fox.mp4', embed=True, width=512))
from google.colab import files; files.download('/content/clip_07_fox.mp4')
